In [ ]:
# Import

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings

warnings.filterwarnings('ignore')

# Matplotlib beállítások
plt.rcParams['figure.figsize'] = (15, 10)
plt.rcParams['font.size'] = 10
sns.set_style("whitegrid")
sns.set_palette("husl")

# Adatok betöltése
df = pd.read_csv('survey_data_cleaned.csv')

In [ ]:
# DEMOGRÁFIAI ÉS ALAPVETŐ JELLEMZŐK

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Alapvető Demográfiai és Fogadási Jellemzők', fontsize=16, fontweight='bold')

# 1.1 Fogadási gyakoriság
freq_counts = df['frequency'].value_counts()
axes[0, 0].bar(range(len(freq_counts)), freq_counts.values, color='steelblue', alpha=0.8)
axes[0, 0].set_xticks(range(len(freq_counts)))
axes[0, 0].set_xticklabels(freq_counts.index, rotation=45, ha='right')
axes[0, 0].set_title('Fogadási gyakoriság', fontweight='bold')
axes[0, 0].set_ylabel('Válaszadók száma')
for i, v in enumerate(freq_counts.values):
    axes[0, 0].text(i, v + 0.1, str(v), ha='center', va='bottom', fontweight='bold')

# 1.2 Tapasztalat (évek)
axes[0, 1].hist(df['years_betting'].dropna(), bins=15, color='coral', alpha=0.7, edgecolor='black')
axes[0, 1].axvline(df['years_betting'].mean(), color='red', linestyle='--', linewidth=2, label=f'Átlag: {df["years_betting"].mean():.1f} év')
axes[0, 1].axvline(df['years_betting'].median(), color='green', linestyle='--', linewidth=2, label=f'Medián: {df["years_betting"].median():.1f} év')
axes[0, 1].set_title('Fogadási tapasztalat (évek)', fontweight='bold')
axes[0, 1].set_xlabel('Évek száma')
axes[0, 1].set_ylabel('Gyakoriság')
axes[0, 1].legend()

# 1.3 Aktivitási szint
activity_counts = df['activity_level'].value_counts()
colors = ['#ff9999', '#ffcc99', '#99ff99']
axes[0, 2].pie(activity_counts.values, labels=activity_counts.index, autopct='%1.1f%%', 
               colors=colors, startangle=90)
axes[0, 2].set_title('Aktivitási szintek', fontweight='bold')

# 1.4 Tapasztalati szint
exp_counts = df['experience_level'].value_counts()
axes[1, 0].barh(range(len(exp_counts)), exp_counts.values, color='mediumpurple', alpha=0.8)
axes[1, 0].set_yticks(range(len(exp_counts)))
axes[1, 0].set_yticklabels(exp_counts.index)
axes[1, 0].set_title('Tapasztalati szintek', fontweight='bold')
axes[1, 0].set_xlabel('Válaszadók száma')
for i, v in enumerate(exp_counts.values):
    axes[1, 0].text(v + 0.1, i, str(v), va='center', fontweight='bold')

# 1.5 Forrás megbízhatósága
reliability = df['source_reliability'].dropna()
axes[1, 1].hist(reliability, bins=5, range=(1, 6), color='teal', alpha=0.7, edgecolor='black')
axes[1, 1].set_title(f'Jelenlegi forrás megbízhatósága\n(Átlag: {reliability.mean():.2f}/5)', fontweight='bold')
axes[1, 1].set_xlabel('Megbízhatósági szint (1-5)')
axes[1, 1].set_ylabel('Gyakoriság')
axes[1, 1].set_xticks(range(1, 6))

# 1.6 Hosszú távú használat hajlandósága
long_term = df['long_term_use'].dropna()
axes[1, 2].hist(long_term, bins=5, range=(1, 6), color='orange', alpha=0.7, edgecolor='black')
axes[1, 2].set_title(f'Hosszú távú használat hajlandósága\n(Átlag: {long_term.mean():.2f}/5)', fontweight='bold')
axes[1, 2].set_xlabel('Hajlandósági szint (1-5)')
axes[1, 2].set_ylabel('Gyakoriság')
axes[1, 2].set_xticks(range(1, 6))

plt.tight_layout()
plt.savefig('01_demographic_overview.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# ÁRKÉPZÉSI ELEMZÉS

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Árképzési Analízis - Willingness to Pay (WTP)', fontsize=16, fontweight='bold')

# 2.1 WTP eloszlás
wtp_data = df['willingness_to_pay'].dropna()
axes[0, 0].hist(wtp_data, bins=20, color='green', alpha=0.6, edgecolor='black')
axes[0, 0].axvline(wtp_data.mean(), color='red', linestyle='--', linewidth=2, 
                   label=f'Átlag: {wtp_data.mean():.0f} Ft')
axes[0, 0].axvline(wtp_data.median(), color='blue', linestyle='--', linewidth=2, 
                   label=f'Medián: {wtp_data.median():.0f} Ft')
axes[0, 0].set_title('Fizetési hajlandóság eloszlása', fontweight='bold')
axes[0, 0].set_xlabel('Fizetési hajlandóság (Ft/hó)')
axes[0, 0].set_ylabel('Gyakoriság')
axes[0, 0].legend()
axes[0, 0].grid(axis='y', alpha=0.3)

# 2.2 WTP boxplot szegmensek szerint
wtp_by_segment = df.groupby('wtp_segment')['willingness_to_pay'].apply(list)
bp = axes[0, 1].boxplot([x for x in wtp_by_segment if len(x) > 0], 
                        labels=[idx for idx, x in wtp_by_segment.items() if len(x) > 0],
                        patch_artist=True)
for patch in bp['boxes']:
    patch.set_facecolor('lightblue')
axes[0, 1].set_title('WTP eloszlás szegmensenként', fontweight='bold')
axes[0, 1].set_ylabel('Fizetési hajlandóság (Ft/hó)')
axes[0, 1].tick_params(axis='x', rotation=45)
axes[0, 1].grid(axis='y', alpha=0.3)

# 2.3 WTP vs Aktivitás
activity_wtp = df.groupby('activity_level')['willingness_to_pay'].agg(['mean', 'median', 'count'])
x_pos = np.arange(len(activity_wtp))
width = 0.35
axes[1, 0].bar(x_pos - width/2, activity_wtp['mean'], width, label='Átlag', color='steelblue', alpha=0.8)
axes[1, 0].bar(x_pos + width/2, activity_wtp['median'], width, label='Medián', color='coral', alpha=0.8)
axes[1, 0].set_xticks(x_pos)
axes[1, 0].set_xticklabels(activity_wtp.index, rotation=45, ha='right')
axes[1, 0].set_title('WTP aktivitási szint szerint', fontweight='bold')
axes[1, 0].set_ylabel('Fizetési hajlandóság (Ft/hó)')
axes[1, 0].legend()
axes[1, 0].grid(axis='y', alpha=0.3)

# Értékek kiírása
for i, (mean_val, median_val) in enumerate(zip(activity_wtp['mean'], activity_wtp['median'])):
    axes[1, 0].text(i - width/2, mean_val + 200, f'{mean_val:.0f}', ha='center', va='bottom', fontsize=9)
    axes[1, 0].text(i + width/2, median_val + 200, f'{median_val:.0f}', ha='center', va='bottom', fontsize=9)

# 2.4 WTP vs Tapasztalat
exp_wtp = df.groupby('experience_level')['willingness_to_pay'].agg(['mean', 'median', 'count'])
x_pos = np.arange(len(exp_wtp))
axes[1, 1].bar(x_pos - width/2, exp_wtp['mean'], width, label='Átlag', color='mediumpurple', alpha=0.8)
axes[1, 1].bar(x_pos + width/2, exp_wtp['median'], width, label='Medián', color='orange', alpha=0.8)
axes[1, 1].set_xticks(x_pos)
axes[1, 1].set_xticklabels(exp_wtp.index, rotation=45, ha='right')
axes[1, 1].set_title('WTP tapasztalati szint szerint', fontweight='bold')
axes[1, 1].set_ylabel('Fizetési hajlandóság (Ft/hó)')
axes[1, 1].legend()
axes[1, 1].grid(axis='y', alpha=0.3)

for i, (mean_val, median_val) in enumerate(zip(exp_wtp['mean'], exp_wtp['median'])):
    axes[1, 1].text(i - width/2, mean_val + 200, f'{mean_val:.0f}', ha='center', va='bottom', fontsize=9)
    axes[1, 1].text(i + width/2, median_val + 200, f'{median_val:.0f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('02_pricing_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# ATTITŰDÖK ÉS VÉLEMÉNYEK

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Attitűdök és Vélemények', fontsize=16, fontweight='bold')

# 3.1 Algoritmus vélemény
algo_counts = df['algo_opinion'].value_counts()
axes[0, 0].barh(range(len(algo_counts)), algo_counts.values, color='skyblue', alpha=0.8)
axes[0, 0].set_yticks(range(len(algo_counts)))
axes[0, 0].set_yticklabels([label[:40] + '...' if len(label) > 40 else label for label in algo_counts.index])
axes[0, 0].set_title('Vélemény az algoritmikus tippekről', fontweight='bold')
axes[0, 0].set_xlabel('Válaszadók száma')
for i, v in enumerate(algo_counts.values):
    axes[0, 0].text(v + 0.1, i, str(v), va='center', fontweight='bold')

# 3.2 Profitgarancia vélemény
guarantee_counts = df['guarantee_opinion'].value_counts()
axes[0, 1].barh(range(len(guarantee_counts)), guarantee_counts.values, color='lightgreen', alpha=0.8)
axes[0, 1].set_yticks(range(len(guarantee_counts)))
axes[0, 1].set_yticklabels([label[:40] + '...' if len(label) > 40 else label for label in guarantee_counts.index])
axes[0, 1].set_title('Vélemény a profitgaranciáról', fontweight='bold')
axes[0, 1].set_xlabel('Válaszadók száma')
for i, v in enumerate(guarantee_counts.values):
    axes[0, 1].text(v + 0.1, i, str(v), va='center', fontweight='bold')

# 3.3 Közösségi preferencia
community_counts = df['community_preference'].value_counts()
colors_comm = ['#ff6b6b', '#4ecdc4', '#45b7d1']
axes[1, 0].pie(community_counts.values, labels=community_counts.index, autopct='%1.1f%%',
               colors=colors_comm, startangle=90)
axes[1, 0].set_title('Közösségi preferencia', fontweight='bold')

# 3.4 Időráfordítás
time_counts = df['time_commitment'].value_counts()
axes[1, 1].bar(range(len(time_counts)), time_counts.values, color='salmon', alpha=0.8)
axes[1, 1].set_xticks(range(len(time_counts)))
axes[1, 1].set_xticklabels(time_counts.index, rotation=45, ha='right')
axes[1, 1].set_title('Napi időráfordítás preferencia', fontweight='bold')
axes[1, 1].set_ylabel('Válaszadók száma')
for i, v in enumerate(time_counts.values):
    axes[1, 1].text(i, v + 0.1, str(v), ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig('03_attitudes_opinions.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# MOTIVÁCIÓK ÉS AKADÁLYOK

# Motivációk feldolgozása (több válasz lehetséges)
all_motivations = []
for motiv in df['motivation'].dropna():
    all_motivations.extend([m.strip() for m in str(motiv).split(',')])

motivation_counts = pd.Series(all_motivations).value_counts()

# Akadályok feldolgozása (több válasz lehetséges)
all_barriers = []
for barrier in df['barriers'].dropna():
    all_barriers.extend([b.strip() for b in str(barrier).split(',')])

barrier_counts = pd.Series(all_barriers).value_counts()

# Csalódások feldolgozása
all_frustrations = []
for frust in df['frustrations'].dropna():
    all_frustrations.extend([f.strip() for f in str(frust).split(',')])

frustration_counts = pd.Series(all_frustrations).value_counts()

fig, axes = plt.subplots(3, 1, figsize=(14, 16))
fig.suptitle('Motivációk, Akadályok és Csalódások', fontsize=16, fontweight='bold')

# 4.1 Motivációk
axes[0].barh(range(len(motivation_counts)), motivation_counts.values, color='gold', alpha=0.8)
axes[0].set_yticks(range(len(motivation_counts)))
axes[0].set_yticklabels([label[:50] + '...' if len(label) > 50 else label for label in motivation_counts.index])
axes[0].set_title('Fogadási motivációk (több válasz lehetséges)', fontweight='bold', fontsize=12)
axes[0].set_xlabel('Említések száma')
for i, v in enumerate(motivation_counts.values):
    axes[0].text(v + 0.1, i, str(v), va='center', fontweight='bold')

# 4.2 Akadályok
axes[1].barh(range(len(barrier_counts)), barrier_counts.values, color='crimson', alpha=0.8)
axes[1].set_yticks(range(len(barrier_counts)))
axes[1].set_yticklabels([label[:50] + '...' if len(label) > 50 else label for label in barrier_counts.index])
axes[1].set_title('Előfizetési akadályok (több válasz lehetséges)', fontweight='bold', fontsize=12)
axes[1].set_xlabel('Említések száma')
for i, v in enumerate(barrier_counts.values):
    axes[1].text(v + 0.1, i, str(v), va='center', fontweight='bold')

# 4.3 Csalódások
axes[2].barh(range(len(frustration_counts)), frustration_counts.values, color='darkorange', alpha=0.8)
axes[2].set_yticks(range(len(frustration_counts)))
axes[2].set_yticklabels([label[:50] + '...' if len(label) > 50 else label for label in frustration_counts.index])
axes[2].set_title('Legnagyobb csalódások (több válasz lehetséges)', fontweight='bold', fontsize=12)
axes[2].set_xlabel('Említések száma')
for i, v in enumerate(frustration_counts.values):
    axes[2].text(v + 0.1, i, str(v), va='center', fontweight='bold')

plt.tight_layout()
plt.savefig('04_motivations_barriers.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# KORRELÁCIÓS ANALÍZIS

# Numerikus változók kiválasztása
numeric_cols = ['years_betting', 'source_reliability', 'willingness_to_pay', 
                'long_term_use', 'frequency_score']
corr_df = df[numeric_cols].dropna()

# Korrelációs mátrix
correlation_matrix = corr_df.corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, 
            square=True, linewidths=1, cbar_kws={"shrink": 0.8}, ax=ax,
            fmt='.2f', vmin=-1, vmax=1)
ax.set_title('Korrelációs mátrix - Kulcsfontosságú változók', fontweight='bold', fontsize=14)

# Címkék olvashatóbbá tétele
labels = ['Évek', 'Forrás\nmegbízhatóság', 'Fizetési\nhajlandóság', 
          'Hosszú távú\nhasználat', 'Fogadási\ngyakoriság']
ax.set_xticklabels(labels, rotation=45, ha='right')
ax.set_yticklabels(labels, rotation=0)

plt.tight_layout()
plt.savefig('05_correlation_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# SCATTER PLOT MÁTRIX - KULCS KAPCSOLATOK

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
fig.suptitle('Kulcsfontosságú Kapcsolatok', fontsize=16, fontweight='bold')

# 6.1 WTP vs Tapasztalat
# Előbb szűrjük ki a NaN értékeket EGYÜTT
valid_data = df[['years_betting', 'willingness_to_pay']].dropna()

axes[0, 0].scatter(valid_data['years_betting'], valid_data['willingness_to_pay'], 
                   alpha=0.6, s=100, c='steelblue')
z = np.polyfit(valid_data['years_betting'], valid_data['willingness_to_pay'], 1)
p = np.poly1d(z)
x_sorted = valid_data['years_betting'].sort_values()
axes[0, 0].plot(x_sorted, p(x_sorted), "r--", linewidth=2, label='Trend')
axes[0, 0].set_xlabel('Fogadási tapasztalat (év)')
axes[0, 0].set_ylabel('Fizetési hajlandóság (Ft/hó)')
axes[0, 0].set_title('WTP vs Tapasztalat', fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# 6.2 WTP vs Hosszú távú használat
axes[0, 1].scatter(df['long_term_use'], df['willingness_to_pay'], alpha=0.6, s=100, c='coral')
axes[0, 1].set_xlabel('Hosszú távú használat hajlandósága (1-5)')
axes[0, 1].set_ylabel('Fizetési hajlandóság (Ft/hó)')
axes[0, 1].set_title('WTP vs Hosszú távú elkötelezettség', fontweight='bold')
axes[0, 1].grid(alpha=0.3)

# 6.3 Forrás megbízhatóság vs WTP
axes[1, 0].scatter(df['source_reliability'], df['willingness_to_pay'], alpha=0.6, s=100, c='green')
axes[1, 0].set_xlabel('Jelenlegi forrás megbízhatósága (1-5)')
axes[1, 0].set_ylabel('Fizetési hajlandóság (Ft/hó)')
axes[1, 0].set_title('WTP vs Forrás megbízhatóság', fontweight='bold')
axes[1, 0].grid(alpha=0.3)

# 6.4 Gyakoriság vs WTP (boxplot)
freq_order = ['Ritkán / csak nagyobb eseményekre', 'Havonta néhányszor', 
              'Hetente többször', 'Naponta']
wtp_by_freq = [df[df['frequency'] == freq]['willingness_to_pay'].dropna().values 
               for freq in freq_order if freq in df['frequency'].values]
bp = axes[1, 1].boxplot(wtp_by_freq, labels=['Ritkán', 'Havonta', 'Hetente', 'Naponta'],
                        patch_artist=True)
for patch in bp['boxes']:
    patch.set_facecolor('lightblue')
axes[1, 1].set_xlabel('Fogadási gyakoriság')
axes[1, 1].set_ylabel('Fizetési hajlandóság (Ft/hó)')
axes[1, 1].set_title('WTP vs Gyakoriság', fontweight='bold')
axes[1, 1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('06_key_relationships.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# ÖSSZEFOGLALÓ STATISZTIKÁK

print("=" * 80)
print("ÖSSZEFOGLALÓ STATISZTIKÁK ÉS INSIGHTS")
print("=" * 80)

print("\n1. ÁRKÉPZÉSI INSIGHTS:")
print(f"   - Átlagos WTP: {df['willingness_to_pay'].mean():.0f} Ft/hó")
print(f"   - Medián WTP: {df['willingness_to_pay'].median():.0f} Ft/hó")
print(f"   - Min-Max WTP: {df['willingness_to_pay'].min():.0f} - {df['willingness_to_pay'].max():.0f} Ft/hó")
print(f"   - Standard deviáció: {df['willingness_to_pay'].std():.0f} Ft")

print("\n2. AKTIVITÁSI SZINTEK WTP-je:")
for level in df['activity_level'].dropna().unique():
    wtp = df[df['activity_level'] == level]['willingness_to_pay']
    print(f"   - {level}: Átlag {wtp.mean():.0f} Ft, Medián {wtp.median():.0f} Ft")

print("\n3. TAPASZTALATI SZINTEK WTP-je:")
for level in df['experience_level'].dropna().unique():
    wtp = df[df['experience_level'] == level]['willingness_to_pay']
    print(f"   - {level}: Átlag {wtp.mean():.0f} Ft, Medián {wtp.median():.0f} Ft")

print("\n4. TOP 3 MOTIVÁCIÓ:")
for i, (motiv, count) in enumerate(motivation_counts.head(3).items(), 1):
    print(f"   {i}. {motiv} ({count} említés)")

print("\n5. TOP 3 AKADÁLY:")
for i, (barrier, count) in enumerate(barrier_counts.head(3).items(), 1):
    print(f"   {i}. {barrier} ({count} említés)")

print("\n6. KORRELÁCIÓK (WTP-vel):")
correlations = correlation_matrix['willingness_to_pay'].drop('willingness_to_pay').sort_values(ascending=False)
for var, corr in correlations.items():
    print(f"   - {var}: {corr:.3f}")

print("\n7. ALGORITMUS ATTITŰD:")
algo_dist = df['algo_opinion'].value_counts(normalize=True) * 100
for opinion, pct in algo_dist.items():
    print(f"   - {opinion}: {pct:.1f}%")

print("\n8. PROFITGARANCIA ATTITŰD:")
guarantee_dist = df['guarantee_opinion'].value_counts(normalize=True) * 100
for opinion, pct in guarantee_dist.items():
    print(f"   - {opinion}: {pct:.1f}%")

print("\n" + "=" * 80)
print("VIZUALIZÁCIÓK ÉS FELTÁRÓ ELEMZÉS BEFEJEZVE!")
print("Mentett képek:")
print("  - 01_demographic_overview.png")
print("  - 02_pricing_analysis.png")
print("  - 03_attitudes_opinions.png")
print("  - 04_motivations_barriers.png")
print("  - 05_correlation_matrix.png")
print("  - 06_key_relationships.png")
print("=" * 80)